# Lab 03: Attention Seq2Seq

**Module 03** | Read `notes/03-attention-seq2seq.pdf` before running this notebook.

Implement additive attention in an encoder-decoder that copies reversed sequences, a classic test bed for alignment weights.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **encoder** | A network that reads the input sequence and produces a representation at each position. |
| **decoder** | A network that generates the output sequence one token at a time, conditioned on the encoder. |
| **attention** | A mechanism that lets the decoder look back at all encoder positions and weight them by relevance. |
| **context vector** | A weighted blend of encoder outputs, customized for each decoder step. |
| **teacher forcing** | During training, feeding the decoder the correct previous token instead of its own prediction. |
| **alignment** | Which input positions the decoder focuses on when producing each output token. |
| **greedy decoding** | At each step, pick the single highest-probability token (no search or sampling). |

Refer back to this table whenever a term appears in code or output.


## What problem are we solving?

Sometimes input and output are **different sequences** (translation, summarization, reversing a string). An **encoder** reads the input; a **decoder** writes the output. Without **attention**, the encoder must squeeze everything into one fixed-size vector, which fails on longer inputs.

## What you will learn

- How encoder-decoder models map one sequence to another
- How **Bahdanau attention** builds a **context vector** for each decoder step
- What **teacher forcing** is and why it speeds up training
- How to read an **attention heatmap** as an **alignment** between input and output
- How **greedy decoding** generates output at inference time

We use a **reverse string** task: easy to verify and produces clear alignment patterns in the heatmap.


### Step 1: The copy task and training pairs

**What this code does:** Defines six source-target pairs (reverse each word), builds a character vocabulary, and adds special tokens (padding, start, end).

**Why we do it:** Reversal is a classic seq2seq test: the first output character should align with the last input character, which makes attention easy to visualize.


**What to look for in the output**

Six pairs listed with 'reverse' as the task. Vocabulary size includes special tokens plus all characters in the pairs.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path("..").resolve()

# Special tokens used in every seq2seq pipeline
PAD, SOS, EOS = 0, 1, 2  # padding, start-of-sequence, end-of-sequence
PAIRS = [
    ("hello", "olleh"),
    ("world", "dlrow"),
    ("ai", "ia"),
    ("seq", "qes"),
    ("copy", "ypoc"),
    ("data", "atad"),
]

# Build character vocabulary (indices start at 3 because 0-2 are special tokens)
chars = sorted({c for src, tgt in PAIRS for s in (src, tgt) for c in s})
char_to_idx = {ch: i + 3 for i, ch in enumerate(chars)}
idx_to_char = {i + 3: ch for i, ch in enumerate(chars)}
vocab_size = len(char_to_idx) + 3


def encode_word(word: str) -> list[int]:
    # Wrap word with SOS at the start and EOS at the end
    end_token = EOS
    return [SOS] + [char_to_idx[c] for c in word] + [end_token]


def decode_ids(ids: list[int]) -> str:
    out = []
    for idx in ids:
        if idx in (PAD, SOS):
            continue
        if idx == EOS:
            break
        out.append(idx_to_char[idx])
    return "".join(out)


src_seqs = [encode_word(s) for s, _ in PAIRS]
tgt_seqs = [encode_word(t) for _, t in PAIRS]
max_src = max(len(s) for s in src_seqs)
max_tgt = max(len(t) for t in tgt_seqs)

print(f"Training pairs: {len(PAIRS)} | Vocab: {vocab_size} | Max src/tgt len: {max_src}/{max_tgt}")
print("\nAll source-target pairs:")
print(f"{'#':<4} {'Source':<10} {'Target':<10} {'Task'}")
print("-" * 40)
for i, (src, tgt) in enumerate(PAIRS):
    print(f"{i:<4} {src:<10} {tgt:<10} reverse")


**Concept: Encoder, decoder, and attention**

The **encoder** reads the full input and stores a hidden vector at each position. The **decoder** generates output one token at a time. **Attention** lets each decoder step ask: "Which encoder positions matter right now?" It computes weights (via softmax), then builds a **context vector** as a weighted sum of encoder outputs.


### Step 2: Encoder-decoder with Bahdanau attention

**What this code does:** Implements an encoder GRU, a Bahdanau attention decoder, and the full Seq2Seq model. Pads batches and creates the model on the active device.

**Why we do it:** This is the core architecture. Understanding how context vectors are built each step is key to reading modern translation and summarization models.


**What to look for in the output**

A printed model tree with Encoder, BahdanauDecoder, and Seq2Seq modules, plus a parameter count.


In [ ]:
EMBED_DIM = 64
HIDDEN_DIM = 128
EPOCHS = 20
LR = 1e-2


class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=PAD)
        self.gru = nn.GRU(EMBED_DIM, HIDDEN_DIM, batch_first=True)

    def forward(self, src: torch.Tensor):
        embedded = self.embed(src)
        outputs, hidden = self.gru(embedded)
        return outputs, hidden  # outputs: one vector per source position


class BahdanauDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=PAD)
        # Decoder input = previous token embedding + context vector
        self.gru = nn.GRU(EMBED_DIM + HIDDEN_DIM, HIDDEN_DIM, batch_first=True)
        # Attention scoring layers (Bahdanau additive attention)
        self.W_enc = nn.Linear(HIDDEN_DIM, HIDDEN_DIM, bias=False)
        self.W_dec = nn.Linear(HIDDEN_DIM, HIDDEN_DIM)
        self.v = nn.Linear(HIDDEN_DIM, 1, bias=False)
        self.out = nn.Linear(HIDDEN_DIM, vocab_size)

    def _attention(self, decoder_hidden, encoder_outputs, src_mask):
        # Score every encoder position against the current decoder state
        dec = self.W_dec(decoder_hidden).unsqueeze(1)
        enc = self.W_enc(encoder_outputs)
        energy = self.v(torch.tanh(dec + enc)).squeeze(-1)
        energy = energy.masked_fill(~src_mask, float("-inf"))  # ignore padding
        weights = F.softmax(energy, dim=-1)  # alignment weights sum to 1
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, weights

    def forward(self, tgt_token, hidden, encoder_outputs, src_mask):
        embedded = self.embed(tgt_token)
        context, weights = self._attention(hidden.squeeze(0), encoder_outputs, src_mask)
        gru_in = torch.cat([embedded, context], dim=-1).unsqueeze(1)
        output, hidden = self.gru(gru_in, hidden)
        logits = self.out(output.squeeze(1))
        return logits, hidden, weights


class Seq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = BahdanauDecoder()

    def forward(self, src, tgt, src_mask):
        enc_out, hidden = self.encoder(src)
        logits_list, attn_list = [], []
        dec_in = tgt[:, 0]  # start with SOS token
        for t in range(1, tgt.size(1)):
            logits, hidden, weights = self.decoder(dec_in, hidden, enc_out, src_mask)
            logits_list.append(logits)
            attn_list.append(weights)
            dec_in = tgt[:, t]  # teacher forcing: use ground-truth previous token
        return torch.stack(logits_list, dim=1), torch.stack(attn_list, dim=1)


def pad_batch(seqs, max_len):
    batch = torch.full((len(seqs), max_len), PAD, dtype=torch.long)
    mask = torch.zeros(len(seqs), max_len, dtype=torch.bool)
    for i, seq in enumerate(seqs):
        batch[i, : len(seq)] = torch.tensor(seq, dtype=torch.long)
        mask[i, : len(seq)] = True
    return batch, mask


src_batch, src_mask = pad_batch(src_seqs, max_src)
tgt_batch, _ = pad_batch(tgt_seqs, max_tgt)
src_batch = src_batch.to(device)
tgt_batch = tgt_batch.to(device)
src_mask = src_mask.to(device)

model = Seq2Seq().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nParameters: {n_params:,}")


**Concept: Teacher forcing**

During training, **teacher forcing** feeds the decoder the **correct** previous token instead of its own (possibly wrong) prediction. This stabilizes learning on small datasets because the decoder always sees good inputs while it is still learning. At inference time we cannot do this, so we use **greedy decoding** (always pick the top token).


### Step 3: Train the seq2seq model

**What this code does:** Runs 20 epochs of training with teacher forcing and plots the loss curve.

**Why we do it:** On six tiny pairs, loss should drop toward zero if the model and attention are working.


**What to look for in the output**

Loss printed at epochs 1, 5, 10, 15, 20, trending downward. The plot should show a falling curve.


In [ ]:
loss_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    logits, _ = model(src_batch, tgt_batch, src_mask)
    # Targets are shifted by one: predict token t given tokens before t
    loss = criterion(logits.reshape(-1, vocab_size), tgt_batch[:, 1:].reshape(-1))
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(f"Epoch {epoch:02d}/{EPOCHS}, loss {loss.item():.4f}")

plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Seq2Seq with Bahdanau attention")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Step 4: Evaluate all pairs

**What this code does:** Runs **greedy decoding** on every training pair and checks whether the predicted string matches the target.

**Why we do it:** Sequence accuracy tells us if the model truly learned reversal, not just lowered training loss.


**What to look for in the output**

Most or all pairs should show 'OK'. Sequence accuracy should be 100% or close on this tiny dataset after training.


In [ ]:
@torch.no_grad()
def decode_greedy(model, src_ids: list[int], max_len: int) -> str:
    model.eval()
    src = torch.tensor([src_ids], dtype=torch.long, device=device)
    mask = torch.ones(1, src.size(1), dtype=torch.bool, device=device)
    enc_out, hidden = model.encoder(src)
    token = torch.tensor([SOS], device=device)
    outputs = []
    for _ in range(max_len):
        logits, hidden, _ = model.decoder(token, hidden, enc_out, mask)
        next_id = logits.argmax(dim=-1)  # greedy: pick highest-probability token
        outputs.append(next_id.item())
        token = next_id
        if next_id.item() == EOS:
            break
    return decode_ids(outputs)


print("Evaluation on all pairs:")
print(f"{'Source':<10} {'Target':<10} {'Predicted':<10} {'Match'}")
print("-" * 44)
correct = 0
for src, tgt in PAIRS:
    src_ids = encode_word(src)
    pred = decode_greedy(model, src_ids, max_tgt)
    match = pred == tgt
    correct += int(match)
    mark = "OK" if match else "WRONG"
    print(f"{src:<10} {tgt:<10} {pred:<10} {mark}")

print(f"\nSequence accuracy: {correct}/{len(PAIRS)} ({100 * correct / len(PAIRS):.0f}%)")


**Concept: Alignment in the attention heatmap**

For string reversal, decoder step *i* should focus on encoder position *n - i*. On the heatmap this appears as a bright **anti-diagonal**. Dark cells on that diagonal mean the model learned **where to look** in the input when writing each output character.


### Step 5: Attention heatmap and alignment interpretation

**What this code does:** Decodes `hello` -> `olleh` while saving attention weights, prints peak encoder indices, and draws a heatmap.

**Why we do it:** The heatmap makes attention concrete. You can literally see which input characters the decoder used for each output step.


**What to look for in the output**

Predicted word should match target. Peak encoder indices should roughly count down. Heatmap should show weight concentrated along the anti-diagonal.


In [ ]:
@torch.no_grad()
def decode_with_attention(model, src_ids, max_len):
    model.eval()
    src = torch.tensor([src_ids], dtype=torch.long, device=device)
    mask = torch.ones(1, src.size(1), dtype=torch.bool, device=device)
    enc_out, hidden = model.encoder(src)
    token = torch.tensor([SOS], device=device)
    outputs, weights = [], []
    for _ in range(max_len):
        logits, hidden, attn = model.decoder(token, hidden, enc_out, mask)
        next_id = logits.argmax(dim=-1)
        outputs.append(next_id.item())
        weights.append(attn.squeeze(0).cpu())
        token = next_id
        if next_id.item() == EOS:
            break
    return outputs, torch.stack(weights)


example_src = src_seqs[0]
pred_ids, attn = decode_with_attention(model, example_src, max_tgt)
src_word, tgt_word = PAIRS[0]
pred_word = decode_ids(pred_ids)

print(f"Source:    {src_word}")
print(f"Target:    {tgt_word}")
print(f"Predicted: {pred_word}")
print(f"\nAttention matrix shape: {tuple(attn.shape)} (decoder steps x encoder positions)")
print("\nAlignment interpretation:")
print("  For string reversal, expect high weights along the anti-diagonal.")
print("  Decoder step 0 should attend to the last source character,")
print("  decoder step 1 to the second-to-last, and so on.")
if attn.numel() > 0:
    peak_enc = attn.argmax(dim=-1).tolist()
    print(f"  Peak encoder index per decoder step: {peak_enc}")

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(attn.numpy(), cmap="Blues", ax=ax, cbar=True)
ax.set_xlabel("Encoder position")
ax.set_ylabel("Decoder step")
ax.set_title(f"Attention weights, '{src_word}' -> '{tgt_word}'")
plt.tight_layout()
plt.show()
